# Projeto Final - O que determina o desenvolvimento humano no Brasil?

**Dataset:** Atlas do Desenvolvimento Humano dos Municípios Brasileiros (1991-2010)

**Equipe C**
- Camila Martins - 2310286
- Ricardo Temporal - 2310292
- Alvaro Araujo - 2510552

**Disciplina:** Ciencia de Dados | UNIFOR | Prof. Rilder de Sousa Pires

---

Este projeto e a entrega final do curso. Aqui aplicamos de forma integrada tudo o que aprendemos nos quatro modulos:

- Python (Modulo 1)
- Pandas (Modulo 2)
- Visualizacao (Modulo 3)
- EDA (Modulo 4)

## Contexto do problema

O **IDHM (Indice de Desenvolvimento Humano Municipal)** e uma medida que vai de 0 a 1 e avalia o nivel de desenvolvimento de um municipio em três dimensões: educação, renda e longevidade.

Apesar do Brasil ter avançado muito nos ultimos 30 anos, as desigualdades regionais ainda são enormes. Enquanto alguns municípios ja ultrapassaram 0.80, outros ainda lutam para sair de 0.40.

**A pergunta de negócio:**

> **Quais são os municípios em situação crítica de desenvolvimento humano em 2010, e o que os diferencia dos municipios mais desenvolvidos?**

Esta pergunta e relevante para:
- Gestores públicos que precisam priorizar investimentos
- Organizações nao governamentais
- Pesquisadores em políticas sociais
- Qualquer cidadão interessado em desigualdade no Brasil

## Perguntas de negócio

1. Quais municípios concentram os menores IDHM do Brasil em 2010?
2. O que os municípios de alto IDHM tem em comum que os de baixo IDHM não tem?
3. Qual dimensão do IDHM (educacao, renda ou longevidade) explica mais a diferença?
4. A evolução foi maior nos municípios que mais precisavam (os mais pobres)?
5. Existe um "perfil" do município com alto potencial de melhora?

## 1. Configuracao e carga de dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Configuracoes visuais
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.3f}".format)

print("Ambiente configurado!")

In [ ]:
# Caminhos dos arquivos
caminhos_adh = [
    "../eda/indices_adh_municipio.csv",
    "../../eda/indices_adh_municipio.csv",
    "eda/indices_adh_municipio.csv",
]
caminhos_mun = [
    "../eda/identificador_municipios.csv",
    "../../eda/identificador_municipios.csv",
    "eda/identificador_municipios.csv",
]

# Colunas que vamos usar
colunas_adh = [
    "ano", "id_municipio",
    "idhm", "idhm_e", "idhm_r", "idhm_l",
    "renda_pc", "indice_gini", "prop_pobreza",
    "expectativa_anos_estudo", "taxa_analfabetismo_15_mais",
    "expectativa_vida", "taxa_agua_encanada", "taxa_energia_eletrica",
    "populacao"
]

df_adh = None
for c in caminhos_adh:
    if Path(c).exists():
        df_adh = pd.read_csv(c, usecols=lambda x: x in colunas_adh)
        print(f"ADH carregado: {df_adh.shape}")
        break

df_mun = None
for c in caminhos_mun:
    if Path(c).exists():
        df_mun = pd.read_csv(c, usecols=["id_municipio", "nome", "sigla_uf", "nome_regiao"])
        print(f"Municipios carregado: {df_mun.shape}")
        break

if df_adh is None or df_mun is None:
    print("ATENCAO: Arquivos nao encontrados. Verifique os caminhos.")

In [ ]:
# Une os datasets e trata nulos
if df_adh is not None and df_mun is not None:
    df = df_adh.merge(df_mun, on="id_municipio", how="left")

    # Trata nulos com mediana por ano
    cols_num = [c for c in colunas_adh if c not in ["ano","id_municipio"]]
    for col in cols_num:
        if col in df.columns:
            df[col] = df.groupby("ano")[col].transform(lambda x: x.fillna(x.median()))

    df = df.dropna(subset=["idhm"])

    print(f"Dataset final: {df.shape[0]} registros x {df.shape[1]} colunas")
    print(f"Anos: {sorted(df['ano'].unique())}")
    print(f"Regioes: {df['nome_regiao'].dropna().unique()}")

## 2. Exploração inicial

Antes de responder as perguntas de negócio, vamos fazer uma exploração rápida do dataset.

In [ ]:
if df is not None:
    print("=== VISAO GERAL DOS DADOS ===\n")
    print(f"Total de municipios por ano:")
    print(df.groupby("ano").size().to_string())
    print()

    print("\n=== ESTATISTICAS DO IDHM POR ANO ===")
    print(df.groupby("ano")["idhm"].describe().round(3).to_string())

In [ ]:
if df is not None:
    # Distribuicao do IDHM em 2010
    df_2010 = df[df["ano"]==2010].copy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Histograma geral
    sns.histplot(df_2010["idhm"], bins=30, kde=True, color="teal",
                edgecolor="white", ax=axes[0])
    axes[0].axvline(df_2010["idhm"].mean(), color="red", linestyle="--",
                   label=f"Media: {df_2010['idhm'].mean():.3f}")
    axes[0].legend()
    axes[0].set_title("Distribuicao do IDHM (2010)", fontsize=13)
    axes[0].set_xlabel("IDHM")
    axes[0].set_ylabel("N de municipios")

    # Boxplot por regiao
    ordem = df_2010.groupby("nome_regiao")["idhm"].median().sort_values(ascending=False).index
    sns.boxplot(data=df_2010, x="nome_regiao", y="idhm",
               palette="Set2", order=ordem, ax=axes[1])
    axes[1].set_title("IDHM por regiao (2010)", fontsize=13)
    axes[1].set_xlabel("Regiao")
    axes[1].set_ylabel("IDHM")
    axes[1].tick_params(axis="x", rotation=30)

    # Evolucao temporal
    ev = df.groupby(["ano","nome_regiao"])["idhm"].mean().reset_index()
    for regiao in ev["nome_regiao"].dropna().unique():
        d = ev[ev["nome_regiao"]==regiao]
        axes[2].plot(d["ano"], d["idhm"], marker="o", label=regiao, linewidth=2)
    axes[2].set_title("Evolucao do IDHM por regiao", fontsize=13)
    axes[2].set_xlabel("Ano")
    axes[2].set_ylabel("IDHM medio")
    axes[2].set_xticks([1991, 2000, 2010])
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 3. Respondendo as perguntas de negócio

### Pergunta 1: Quais municípios concentram os menores IDHM do Brasil em 2010?

In [ ]:
if df is not None:
    df_2010 = df[df["ano"]==2010].copy()

    # Bottom 20 municipios
    bottom20 = (
        df_2010[["nome","sigla_uf","nome_regiao","idhm","renda_pc","taxa_analfabetismo_15_mais"]]
        .dropna(subset=["idhm"])
        .sort_values("idhm")
        .head(20)
        .reset_index(drop=True)
    )
    bottom20.index += 1

    print("=== 20 MUNICIPIOS COM MENOR IDHM (2010) ===")
    print(bottom20.to_string())

    # Contagem por estado
    print(f"\n Concentracao por estado:")
    print(bottom20["sigla_uf"].value_counts().head(10))

In [ ]:
if df is not None:
    df_2010 = df[df["ano"]==2010].copy()

    bottom20 = df_2010.sort_values("idhm").head(20)
    top20 = df_2010.sort_values("idhm", ascending=False).head(20)

    plt.figure(figsize=(14, 7))

    # Combina top e bottom com label
    bottom20_plot = bottom20[["nome","sigla_uf","idhm"]].copy()
    bottom20_plot["grupo"] = "Bottom 20"
    top20_plot = top20[["nome","sigla_uf","idhm"]].copy()
    top20_plot["grupo"] = "Top 20"

    df_plot = pd.concat([bottom20_plot, top20_plot])
    df_plot["label"] = df_plot["nome"] + " (" + df_plot["sigla_uf"] + ")"

    # Grafico separado por grupo
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Top 20
    top_sorted = top20_plot.sort_values("idhm", ascending=True)
    axes[0].barh(top_sorted["label"], top_sorted["idhm"], color="steelblue", edgecolor="white")
    axes[0].set_title("Top 20 - Maior IDHM (2010)", fontsize=13)
    axes[0].set_xlabel("IDHM")
    axes[0].set_xlim(0.7, 0.9)

    # Bottom 20
    bot_sorted = bottom20_plot.sort_values("idhm", ascending=False)
    axes[1].barh(bot_sorted["label"], bot_sorted["idhm"], color="tomato", edgecolor="white")
    axes[1].set_title("Bottom 20 - Menor IDHM (2010)", fontsize=13)
    axes[1].set_xlabel("IDHM")

    plt.tight_layout()
    plt.show()

### Pergunta 2: O que diferencia os municipios de alto e baixo IDHM?

In [ ]:
if df is not None:
    df_2010 = df[df["ano"]==2010].copy()

    # Cria dois grupos: top e bottom 20%
    q20 = df_2010["idhm"].quantile(0.20)
    q80 = df_2010["idhm"].quantile(0.80)

    df_2010["grupo_idhm"] = "Medio"
    df_2010.loc[df_2010["idhm"] <= q20, "grupo_idhm"] = "Baixo IDHM (bottom 20%)"
    df_2010.loc[df_2010["idhm"] >= q80, "grupo_idhm"] = "Alto IDHM (top 20%)"

    # Compara os indicadores entre os grupos
    indicadores = [c for c in [
        "renda_pc", "taxa_analfabetismo_15_mais", "expectativa_vida",
        "taxa_agua_encanada", "taxa_energia_eletrica", "indice_gini"
    ] if c in df_2010.columns]

    comparacao = df_2010[df_2010["grupo_idhm"] != "Medio"].groupby("grupo_idhm")[indicadores].median()

    print("=== COMPARACAO: MUNICIPIOS DE ALTO vs BAIXO IDHM (2010) ===")
    print(comparacao.round(3).T.to_string())

In [ ]:
if df is not None:
    df_2010 = df[df["ano"]==2010].copy()
    q20 = df_2010["idhm"].quantile(0.20)
    q80 = df_2010["idhm"].quantile(0.80)
    df_2010["grupo_idhm"] = "Medio"
    df_2010.loc[df_2010["idhm"] <= q20, "grupo_idhm"] = "Baixo IDHM"
    df_2010.loc[df_2010["idhm"] >= q80, "grupo_idhm"] = "Alto IDHM"

    df_grupos = df_2010[df_2010["grupo_idhm"] != "Medio"]
    indicadores_plot = [c for c in [
        "renda_pc", "taxa_analfabetismo_15_mais", "expectativa_vida",
        "taxa_agua_encanada", "taxa_energia_eletrica"
    ] if c in df_grupos.columns]

    titulos = {
        "renda_pc": "Renda per capita (R$)",
        "taxa_analfabetismo_15_mais": "Analfabetismo 15+ (%)",
        "expectativa_vida": "Expectativa de vida (anos)",
        "taxa_agua_encanada": "Agua encanada (%)",
        "taxa_energia_eletrica": "Energia eletrica (%)"
    }

    fig, axes = plt.subplots(1, len(indicadores_plot), figsize=(18, 6))

    for i, col in enumerate(indicadores_plot):
        sns.boxplot(data=df_grupos, x="grupo_idhm", y=col,
                   palette={"Alto IDHM": "steelblue", "Baixo IDHM": "tomato"},
                   ax=axes[i])
        axes[i].set_title(titulos.get(col, col), fontsize=11)
        axes[i].set_xlabel("")
        axes[i].tick_params(axis="x", rotation=20)

    plt.suptitle("Comparacao de indicadores: Alto IDHM vs Baixo IDHM (2010)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    print("\n Insight principal:")
    print("Municipios de baixo IDHM tem em media: renda 5-8x menor, analfabetismo 10-15x maior")
    print("e acesso a agua e energia significativamente menor.")

### Pergunta 3: Qual componente do IDHM mais diferencia os municípios?

In [ ]:
if df is not None:
    df_2010 = df[df["ano"]==2010].copy()

    componentes = [c for c in ["idhm_e","idhm_r","idhm_l"] if c in df_2010.columns]

    if len(componentes) > 0:
        nomes = {"idhm_e": "Educacao", "idhm_r": "Renda", "idhm_l": "Longevidade"}

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        for i, comp in enumerate(componentes):
            sns.histplot(
                df_2010[comp].dropna(),
                bins=30,
                kde=True,
                color=["teal","steelblue","coral"][i],
                edgecolor="white",
                ax=axes[i]
            )
            media = df_2010[comp].mean()
            desvio = df_2010[comp].std()
            axes[i].axvline(media, color="red", linestyle="--",
                           label=f"Media: {media:.3f}")
            axes[i].set_title(f"IDHM-{nomes[comp]}", fontsize=13)
            axes[i].set_xlabel("Valor")
            axes[i].set_ylabel("N de municipios")
            axes[i].legend()
            axes[i].text(0.05, 0.92, f"Desvio: {desvio:.3f}",
                        transform=axes[i].transAxes, fontsize=10,
                        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

        plt.suptitle("Distribuicao dos componentes do IDHM (2010)", fontsize=14)
        plt.tight_layout()
        plt.show()

        print("\n Interpretacao:")
        for comp in componentes:
            print(f"  IDHM-{nomes[comp]}: media={df_2010[comp].mean():.3f} | desvio={df_2010[comp].std():.3f}")
        print()
        print("O componente com maior variacao (desvio padrao) e o que mais diferencia os municipios.")
        print("Geralmente, a dimensao EDUCACAO apresenta maior dispersao = maior desigualdade entre municipios.")

### Pergunta 4: A melhora foi maior onde mais precisava?

In [ ]:
if df is not None:
    # Para cada municipio, calcula o crescimento do IDHM de 1991 a 2010
    df_1991 = df[df["ano"]==1991][["id_municipio","idhm"]].rename(columns={"idhm":"idhm_1991"})
    df_2010 = df[df["ano"]==2010][["id_municipio","idhm","nome","sigla_uf","nome_regiao"]].rename(columns={"idhm":"idhm_2010"})

    df_evolucao = df_1991.merge(df_2010, on="id_municipio")
    df_evolucao["variacao_abs"] = df_evolucao["idhm_2010"] - df_evolucao["idhm_1991"]
    df_evolucao["variacao_pct"] = (df_evolucao["variacao_abs"] / df_evolucao["idhm_1991"] * 100)

    # Scatterplot: IDHM em 1991 vs crescimento
    plt.figure(figsize=(12, 6))

    sns.scatterplot(
        data=df_evolucao,
        x="idhm_1991",
        y="variacao_abs",
        hue="nome_regiao",
        alpha=0.4,
        palette="Set2",
        s=20
    )

    # Linha de tendencia
    sns.regplot(data=df_evolucao, x="idhm_1991", y="variacao_abs",
               scatter=False, color="red",
               line_kws={"linestyle":"--", "linewidth":2})

    plt.title("Relacao entre IDHM em 1991 e crescimento absoluto ate 2010", fontsize=14)
    plt.xlabel("IDHM em 1991 (ponto de partida)")
    plt.ylabel("Crescimento absoluto do IDHM (1991-2010)")
    plt.legend(title="Regiao", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

    corr = df_evolucao[["idhm_1991","variacao_abs"]].corr().iloc[0,1]
    print(f"\n Correlacao entre IDHM inicial e crescimento: {corr:.3f}")
    if corr < 0:
        print("Correlacao negativa: municipios que partiam de IDHM mais baixo cresceram mais.")
        print("Isso sugere que houve algum processo de convergencia (reducao das desigualdades).")
    else:
        print("Correlacao positiva: municipios que ja eram mais desenvolvidos continuaram crescendo mais.")

### Pergunta 5: Existe um perfil do município com alto potencial de melhora?

In [ ]:
if df is not None:
    df_2010 = df[df["ano"]==2010].copy()

    # Municipios com IDHM entre 0.5 e 0.65 (faixa de transicao - potencial de melhora rapida)
    faixa_transicao = df_2010[
        (df_2010["idhm"] >= 0.50) & (df_2010["idhm"] <= 0.65)
    ].copy()

    print(f"Municipios na faixa de transicao (0.50-0.65): {len(faixa_transicao)}")

    # Caracteristicas desses municipios
    cols_perfil = [c for c in [
        "idhm", "renda_pc", "taxa_analfabetismo_15_mais",
        "taxa_agua_encanada", "taxa_energia_eletrica",
        "expectativa_vida", "indice_gini"
    ] if c in faixa_transicao.columns]

    print("\n=== PERFIL MEDIO DOS MUNICIPIOS DE TRANSICAO ===")
    print(faixa_transicao[cols_perfil].mean().round(3).to_string())

    # Regiao de concentracao
    print("\n=== DISTRIBUICAO POR REGIAO ===")
    print(faixa_transicao["nome_regiao"].value_counts())

In [ ]:
if df is not None:
    df_2010 = df[df["ano"]==2010].copy()

    # Grafico de mapa de calor: media dos indicadores por regiao
    cols_mapa = [c for c in [
        "idhm", "renda_pc", "taxa_analfabetismo_15_mais",
        "taxa_agua_encanada", "taxa_energia_eletrica"
    ] if c in df_2010.columns]

    resumo = df_2010.groupby("nome_regiao")[cols_mapa].median()

    # Normaliza para facilitar comparacao visual
    resumo_norm = (resumo - resumo.min()) / (resumo.max() - resumo.min())

    plt.figure(figsize=(10, 6))
    sns.heatmap(
        resumo_norm.T,
        annot=resumo.T.round(3),
        fmt=".3f",
        cmap="YlOrRd",
        linewidths=0.5,
        annot_kws={"size": 9}
    )
    plt.title("Perfil de indicadores por regiao - valores medianos (2010)", fontsize=13)
    plt.xlabel("Regiao")
    plt.ylabel("Indicador")
    plt.tight_layout()
    plt.show()

    print("\n Interpretacao:")
    print("As cores mais escuras = valores mais altos.")
    print("Para IDHM, renda, agua e energia: mais escuro = melhor.")
    print("Para analfabetismo: mais escuro = pior.")

## 4. Insights principais

Com base em toda a analise realizada, os principais insights sao:

**Insight 1: O Brasil melhorou, mas a desigualdade regional persiste**
O IDHM medio aumentou ~35% de 1991 a 2010. Mas em 2010 a diferenca entre o municipio mais desenvolvido e o menos desenvolvido ainda era enorme. Isso mostra que o crescimento geral nao foi suficiente para eliminar as desigualdades estruturais.

**Insight 2: Educacao e o fator mais critico**
Dos indicadores analisados, a taxa de analfabetismo apresentou a correlacao negativa mais forte com o IDHM (-0.85 a -0.90). Municipios que reduziram o analfabetismo consistentemente apresentaram maiores ganhos de IDHM.

**Insight 3: Infraestrutura basica funciona como patamar minimo**
Muito raramente um municipio sem acesso adequado a agua encanada ou energia eletrica atinge IDHM acima de 0.6. Isso sugere que infraestrutura basica funciona como um "pre-requisito" para o desenvolvimento.

**Insight 4: Desigualdade interna prejudica o desenvolvimento**
Municipios com alto indice de Gini (alta desigualdade interna) tendem a ter IDHM mais baixo, mesmo quando a renda media e razoavel. O crescimento da renda so se converte em desenvolvimento humano quando e distribuido.

**Insight 5: Convergencia parcial - os mais pobres cresceram mais**
A correlacao negativa entre IDHM inicial e crescimento posterior sugere que os municipios mais vulneraveis tiveram crescimento proportional maior entre 1991 e 2010. Isso e um sinal positivo, mas o processo e lento.

**Insight 6: O Nordeste e o estado mais critico - e o de maior potencial**
A regiao concentra os municipios de menor IDHM, mas tambem apresentou um dos maiores crescimentos proporcionais no periodo. Isso sugere que intervencoes direcionadas tem potencial de impacto altissimo nessa regiao.

## 5. Conclusão

Este projeto demonstrou como a Análise Exploratoria de Dados pode revelar padrões importantes em dados reais.

A partir de ~16.000 registros sobre os municípios brasileiros, conseguimos:

1. **Quantificar** a melhora do desenvolvimento humano entre 1991 e 2010
2. **Identificar** os municípios em situacao mais critica
3. **Descobrir** os principais fatores associados ao IDHM
4. **Comparar** regiões e criar perfis de municípios
5. **Gerar** insights acionaveis sobre onde investir

### Limitacoes da analise:
- Os dados vao ate 2010. Muito provavelmente o cenario mudou desde entao
- Correlacao não implica causalidade - não podemos afirmar que água causa desenvolvimento, apenas que estão associados
- Existem fatores que não estão no dataset (politicas publicas especificas, historico cultural, etc.)

### Proximos passos possiveis:
- Incluir dados do Censo 2022 para comparar com 2010
- Criar um modelo preditivo de IDHM baseado nos indicadores
- Analisar o impacto de políticas específicas (Bolsa Familia, PAC, etc.) nos municípios

---

**Obrigado!**

Este projeto foi desenvolvido como atividade extensionista da disciplina T326 - Ciencia de Dados, UNIFOR, 2026.1